Part 1 — Dataset preparation

In [0]:
# INSTRUCTOR SETUP

Step 1 — Create the source Delta table

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),

(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


Step 2 — Create a working Delta table

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


Step 3 — Create the zone master table

In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


Step 4 — Prepare incoming records for MERGE

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

Part 2 — Assessment questions

Section A — SQL and DataFrames (20 marks)

In [0]:
%sql
--1
select session_id, vehicle_type, parking_fee
from parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
--2
select *
from parking_sessions_source
where status = 'Completed' and parking_fee  > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
select 
    vehicle_type, 
    sum(parking_fee) as total_parking_fee,
    count(*) as total_ompleted_sessions
from parking_sessions_source
where status = 'Completed'
group by vehicle_type;

vehicle_type,total_parking_fee,total_ompleted_sessions
Bike,160,4
SUV,540,2
Car,1750,8


In [0]:
#4
pyspark_df = spark.sql('select * from parking_sessions_source')
pyspark_df.show()

+----------+-------+------------+--------------+-----------+---------+
|session_id|zone_id|vehicle_type|duration_hours|parking_fee|   status|
+----------+-------+------------+--------------+-----------+---------+
|       101|    Z01|         Car|             3|        150|Completed|
|       102|    Z02|        Bike|             2|         40|Completed|
|       103|    Z03|         Car|             5|        250|Completed|
|       104|    Z01|         SUV|             4|        240|   Active|
|       105|    Z04|        Bike|             1|         20|Completed|
|       106|    Z02|         Car|             6|        300|Completed|
|       107|    Z03|         SUV|             2|        120|Cancelled|
|       108|    Z05|         Car|             3|        150|Completed|
|       109|    Z01|        Bike|             4|         80|Completed|
|       110|    Z04|         SUV|             5|        300|   Active|
|       111|    Z02|         Car|             2|        100|Completed|
|     

In [0]:
from pyspark.sql.functions import col,when
pyspark_df = pyspark_df.withColumn(
    'fee_category',
    when(col('parking_fee') >= 200, 'High')
    .otherwise('Low')
)
display(pyspark_df)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


Section B — Delta Tables and CRUD (20 marks)

In [0]:
%sql
--5
create or replace table parking_summary
using delta 
as
select *
from parking_sessions_source
where status = 'Completed';


num_affected_rows,num_inserted_rows


In [0]:
%sql
-- to display contents
select * from parking_summary;


session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed


In [0]:
%sql
-- to check whether the created table is a delta table
describe detail parking_summary;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,00b99947-ea86-4645-a74d-e43d455b9224,ettl_training_part2.default.parking_summary,null,abfss://unity-catalog-storage@dbstoragez5uw476d45jm2.dfs.core.windows.net/7405618872645793/__unitystorage/catalogs/15dd880d-6a12-43f2-9ef2-18d6a327589e/tables/4e465a67-efa9-45d8-9ebe-55f1938ee05b,2026-11-06T09:04:43.947Z,2026-11-06T09:04:45Z,List(),List(),1,2241,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 4e465a67-efa9-45d8-9ebe-55f1938ee05b, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-c904a329-ca3b-4342-8a36-ed5f93126e9d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-6581ddfe-41c0-40a7-9e05-33b1ab59cd7a, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, catalogManaged, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint, vacuumProtocolCheck)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
--6
insert into parking_sessions_work
values
(123,'Z01','Car', 3, 150, 'Active');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
--7
update parking_sessions_work
set 
    status = 'Completed',
    parking_fee = 280
where session_id = 104;

num_affected_rows
1


In [0]:
%sql
--8
delete from parking_sessions_work
where status = 'Cancelled';

num_affected_rows
2


In [0]:

%sql
select * 
from parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


Section C — MERGE and Delta Operations (20 marks)

In [0]:
%sql
--9 Display incoming_parking_sessions
select *
from incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
--9 common records in both incoming_parking_sessions and prking_sessions_work
select *
from incoming_parking_sessions ips    
inner join parking_sessions_work psw 
on ips.session_id = psw.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed,102,Z02,Bike,2,40,Completed
104,Z01,SUV,5,300,Completed,104,Z01,SUV,4,280,Completed


In [0]:
%sql
--10
merge into parking_sessions_work psw 
using incoming_parking_sessions ips 
on psw.session_id = ips.session_id 
when matched then 
update set 
    psw.zone_id = ips.zone_id,
    psw.vehicle_type = ips.vehicle_type,
    psw.duration_hours = ips.duration_hours,
    psw.parking_fee = ips.parking_fee,
    psw.status = ips.status
when not matched then
insert (zone_id, vehicle_type, duration_hours, parking_fee, status)
values (ips.zone_id, ips.vehicle_type, ips.duration_hours, ips.parking_fee, ips.status);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
--11
describe history parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-11-06T09:05:47.974Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(session_id#3118 = session_id#3100)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(2890365650097471),null,1106-041315-gg7csiib,11,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 10381, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 11522, materializeSourceTimeMs -> 693, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3919, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 6737)",null,Databricks-Runtime/18.x-photon-scala2.13
11,2026-11-06T09:05:32.602Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2890365650097471),null,1106-041315-gg7csiib,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6823, p25FileSize -> 3121, numDeletionVectorsRemoved -> 1, minFileSize -> 3121, numAddedFiles -> 1, maxFileSize -> 3121, p75FileSize -> 3121, p50FileSize -> 3121, numAddedBytes -> 3121)",null,Databricks-Runtime/18.x-photon-scala2.13
10,2026-11-06T09:05:20.441Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#2367 = Cancelled)""])",null,List(2890365650097471),null,1106-041315-gg7csiib,9,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 6022, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 3557, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 2454)",null,Databricks-Runtime/18.x-photon-scala2.13
9,2026-11-06T09:05:11.382Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#1874 = 104)""])",null,List(2890365650097471),null,1106-041315-gg7csiib,8,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 12619, numDeletionVectorsUpdated -> 0, scanTimeMs -> 5862, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2655, rewriteTimeMs -> 6699)",null,Databricks-Runtime/18.x-photon-scala2.13
8,2026-11-06T09:04:56.103Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(2890365650097471),null,1106-041315-gg7csiib,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1866)",null,Databricks-Runtime/18.x-photon-scala2.13
7,2026-11-06T09:04:05.1Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databri

In [0]:
%sql
--12
select 'Before_merage_operations' as operation, *
from parking_sessions_work version as of 4
where session_id = 102

union all

select 'After_merage_operations' as operation, *
from parking_sessions_work
where session_id = 102;

operation,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
Before_merage_operations,102,Z02,Bike,2,40,Completed
After_merage_operations,102,Z02,Bike,3,60,Completed


Section D — Tables, Views and Optimization (20 marks)

In [0]:
%sql
--13
create or replace view completed_parking_view
as
select *
from parking_sessions_work
where status = 'Completed';

In [0]:
%sql
select * from completed_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


In [0]:
%sql
--14
create or replace temp view temp_Z01_sessions
as
select *
from parking_sessions_work
where zone_id = 'Z01';

In [0]:
%sql
create or replace global temporary view parking_fee_above_200
as 
select *
from parking_sessions_work
where parking_fee > 200;

In [0]:
%sql
--15
select 
    pz.zone_name,
    pz.city,
    psw.vehicle_type,
    psw.parking_fee
from parking_sessions_work psw 
join parking_zones pz 
on psw.zone_id = pz.zone_id;

zone_name,city,vehicle_type,parking_fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
%sql
--16
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2c3db770-ac8b-4101-8933-a23ef5881105,ettl_training_part2.default.parking_sessions_work,null,abfss://unity-catalog-storage@dbstoragez5uw476d45jm2.dfs.core.windows.net/7405618872645793/__unitystorage/catalogs/15dd880d-6a12-43f2-9ef2-18d6a327589e/tables/b06c11f6-d6e2-4c3d-ab0b-cf82b73e6da9,2026-11-06T09:04:01.325Z,2026-11-06T09:05:55Z,List(),List(),1,3124,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b06c11f6-d6e2-4c3d-ab0b-cf82b73e6da9, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-ebe0ac5c-01ac-4deb-893f-635cfdb1ad06, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-ad458e17-170f-479e-948d-d4bd6eeb23f3, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, catalogManaged, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint, vacuumProtocolCheck)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
optimize parking_sessions_work;

path,metrics
abfss://unity-catalog-storage@dbstoragez5uw476d45jm2.dfs.core.windows.net/7405618872645793/__unitystorage/catalogs/15dd880d-6a12-43f2-9ef2-18d6a327589e/tables/b06c11f6-d6e2-4c3d-ab0b-cf82b73e6da9,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793955975907, 1793955977516, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null)"


Section E — Structured Streaming and DLT (20 marks)

In [0]:
import time
from pyspark.sql import SparkSession

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("RateSourceExample") \
    .getOrCreate()

# 2. Read stream from the built-in rate source
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# 3. Display the streaming data
query = streaming_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "/Volumes/ettl_training_part2/default/checkpoints/rate_demo"
    ) \
    .start()

# 4. Run the stream for 30 seconds
try:
    time.sleep(30)
finally:
    query.stop()

In [0]:
#18
from pyspark.sql.functions import col

streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

transformed_df = streaming_df.withColumn(
    'amount',
    col('value')*10
).filter(
    col('amount') > 300
)
display(transformed_df, checkpointLocation = "/Volumes/ettl_training_part2/default/checkpoints/rate_transformed")

timestamp,value,amount
2026-11-06T09:07:02.811Z,32,320
2026-11-06T09:07:03.211Z,34,340
2026-11-06T09:07:03.611Z,36,360
2026-11-06T09:07:03.011Z,33,330
2026-11-06T09:07:03.411Z,35,350
2026-11-06T09:07:03.811Z,37,370
2026-11-06T09:07:02.611Z,31,310
2026-11-06T09:07:04.411Z,40,400
2026-11-06T09:07:04.811Z,42,420
2026-11-06T09:07:05.211Z,44,440
